# 22DM015 Final Project — Financial PhraseBank
**Part 4 Model Distillation / Quantization**

## Preparations

In [26]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys, logging, warnings, time, tempfile
# Environment + logging setup that MUST run BEFORE importing transformers/torch.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # let unsupported MPS ops fall back to CPU
for name in ("tensorflow", "torch.distributed.elastic.multiprocessing.redirects", "torchao"):
    logging.getLogger(name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=r".*Skipping import of cpp extensions.*")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, set_seed, EarlyStoppingCallback, default_data_collator)
import transformers
transformers.logging.set_verbosity_error()       # hide the per-load model LOAD REPORTs
transformers.logging.disable_progress_bar()       # drop shard-save / Map bars (keep the metrics TABLE)
import datasets.utils.logging as _ds_logging
_ds_logging.disable_progress_bar()

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath(".."))
import data_utils as du
import eval_utils as eu

SEED = du.SEED
random.seed(SEED); np.random.seed(SEED); set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

Last run: 2026-06-17 12:06:05


In [27]:
# watermark: AGLLM (AI-assisted content disclosure)
splits = du.load_splits()            # identical train/val/test for everyone
train, val, test = splits["train"], splits["val"], splits["test"]

# Part 4 is a COMPARISON of compression methods, not a push for peak accuracy, so for speed the
# base model and the distilled model are trained on a 50% stratified slice of train. val/test
# stay the full held-out splits (identical to Part 2/3), so the numbers remain comparable.
TRAIN_FRAC = 0.5
train_sub = du.subset_by_fraction(train, TRAIN_FRAC)   # stratified; preserves class proportions
print(f"train(full)={len(train)}  train_sub({int(TRAIN_FRAC*100)}%)={len(train_sub)}  "
      f"val={len(val)}  test={len(test)}")

train(full)=1584  train_sub(50%)=792  val=227  test=453
Last run: 2026-06-17 12:06:07


In [7]:
# watermark: AGLLM (AI-assisted content disclosure)
# Device: CUDA > MPS (Apple Silicon / M4) > CPU. Training runs on DEVICE; compression and the
# latency profiling run on CPU (the deployment target the assignment cares about).
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"torch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 4)

BERT_ID = "bert-base-uncased"             # the classifier we compress
DISTILBERT_ID = "distilbert-base-uncased" # 6 layers vs 12; shares BERT's vocab/tokenizer
BERT_DIR = "../.cache/bert_p4"
DISTILBERT_DIR = "../.cache/distilbert"
NUM_LABELS = 3

# Training hyperparameters. Part 4 is a COMPARISON of compression methods, so we use a lighter
# (faster) protocol than Part 2/3's deliverable run -- shorter sequences, bigger batches, fewer
# epochs -- applied IDENTICALLY to bert-base and DistilBERT, so the comparison stays fair. We
# keep the core method: full fine-tuning with early stopping on val macro-F1 and the live table.
MAX_LEN = 64                  # Financial PhraseBank sentences fit in 64 tokens (Part 2/3 used 128)
BATCH = 16                    # bigger batch -> fewer steps (Part 2/3 used 8)
EPOCHS = 8                    # upper bound; early stopping usually stops well before this (was 20)
EARLY_STOP_PATIENCE = 2       # epochs without val macro-F1 gain before stopping (Part 2/3 used 3)
ALPHA, TEMPERATURE = 0.5, 2.0 # KD loss: alpha*CE + (1-alpha)*T^2*KL

# One tokenizer for both: DistilBERT uses the bert-base-uncased vocabulary, so the same token
# ids feed both and the distillation soft targets line up directly.
tok = AutoTokenizer.from_pretrained(BERT_ID)

torch 2.12.0 | device: mps
Last run: 2026-06-17 10:02:11


In [18]:
# watermark: AGLLM (AI-assisted content disclosure)
# Shared helpers: tokenisation, the training protocol (with optional KD), and the compression
# profiling (params / on-disk size / single-thread CPU latency / training runtime).
def encode(df, max_len=MAX_LEN):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    ds = ds.map(lambda b: tok(b["text"], truncation=True, padding="max_length",
                              max_length=max_len, return_token_type_ids=False), batched=True)
    return ds.rename_column("label", "labels").remove_columns(["text"])


def compute_metrics(eval_pred):
    # macro-F1 (logged as eval_f1_macro) drives best-checkpoint selection during early stopping.
    return eu.evaluate(eval_pred.label_ids, eval_pred.predictions.argmax(-1))


class DistillTrainer(Trainer):
    """Trainer whose loss is plain cross-entropy, or -- when `bert_model` is given -- a KD blend
    alpha*CE(distilbert, gold) + (1-alpha)*T^2*KL(distilbert_T || bert_T). bert_model is the
    fixed source of the soft targets (kept in eval mode, no grad)."""
    def __init__(self, bert_model=None, alpha=ALPHA, temperature=TEMPERATURE, **kw):
        super().__init__(**kw)
        self.bert = bert_model.eval() if bert_model is not None else None
        self.alpha, self.T = alpha, temperature

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs["labels"]
        out = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        ce = F.cross_entropy(out.logits, labels)
        if self.bert is None:                              # plain fine-tuning (the base bert-base)
            loss = ce
        else:                                              # knowledge distillation (the student)
            with torch.no_grad():
                bert_logits = self.bert(input_ids=inputs["input_ids"],
                                        attention_mask=inputs["attention_mask"]).logits
            kd = F.kl_div(F.log_softmax(out.logits / self.T, dim=-1),
                          F.softmax(bert_logits / self.T, dim=-1),
                          reduction="batchmean") * (self.T ** 2)
            loss = self.alpha * ce + (1 - self.alpha) * kd
        return (loss, out) if return_outputs else loss


def train_model(model_id, train_df, out_dir, *, val_df=None, teacher=None, seed=SEED, epochs=EPOCHS,
                batch=BATCH, max_len=MAX_LEN, lr=2e-5, log_epochs=True, patience=EARLY_STOP_PATIENCE):
    """Full fine-tuning with early stopping on val macro-F1, using the lighter Part-4 protocol
    set in the hyperparameter cell (max_len 64, batch 16, up to 8 epochs, patience 2), the live
    metrics TABLE, on the detected DEVICE (CUDA / MPS / CPU). teacher given -> knowledge
    distillation via DistillTrainer. `seed` controls weight-init + data shuffling, so passing
    different seeds gives independent training runs (used by the variance analysis).
    Returns (best-on-val model on CPU, train_runtime_seconds) -- the runtime is HF's measured
    training wall-clock, so the cost of each model is recorded for the 4b comparison."""
    val_df = val if val_df is None else val_df
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_LABELS)
    if teacher is not None:
        teacher.to(DEVICE)                         # soft-target model on the same device as inputs
    args = TrainingArguments(
        output_dir=out_dir + "/_hf", seed=seed, num_train_epochs=epochs,
        per_device_train_batch_size=batch, per_device_eval_batch_size=64, learning_rate=lr,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        logging_strategy="epoch" if log_epochs else "no",
        report_to="none", disable_tqdm=False,      # keep the live notebook metrics TABLE (as Part 2/3)
        use_cpu=(DEVICE == "cpu"), dataloader_pin_memory=(DEVICE == "cuda"),
    )
    trainer = DistillTrainer(
        bert_model=teacher, model=model, args=args, data_collator=default_data_collator,
        train_dataset=encode(train_df, max_len), eval_dataset=encode(val_df, max_len),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=patience)],
    )
    out = trainer.train()
    if teacher is not None:
        teacher.to("cpu")
    runtime_s = float(out.metrics.get("train_runtime", float("nan")))   # HF-measured training wall-clock
    return trainer.model.to("cpu"), runtime_s      # CPU for compression + latency profiling


@torch.no_grad()
def predict(model, df, max_len=MAX_LEN, batch=64):
    """argmax predictions on df, on CPU (the deployment target), using input_ids/attention_mask."""
    model.to("cpu").eval()
    ds = encode(df, max_len)
    ids, mask = torch.tensor(ds["input_ids"]), torch.tensor(ds["attention_mask"])
    out = []
    for i in range(0, len(ds), batch):
        out.append(model(input_ids=ids[i:i + batch], attention_mask=mask[i:i + batch]).logits.argmax(-1))
    return torch.cat(out).numpy()


def score(model, df=test):
    return eu.evaluate(df["label"].values, predict(model, df))


def n_params_m(model):
    return sum(p.numel() for p in model.parameters()) / 1e6


def size_mb(model):
    """On-disk size of the serialised state_dict in MB (reflects int8 for quantised models)."""
    f = tempfile.NamedTemporaryFile(suffix=".pt", delete=False); f.close()
    torch.save(model.state_dict(), f.name)
    mb = os.path.getsize(f.name) / 1e6
    os.remove(f.name)
    return mb


def latency_ms(model, df=test, n=80, threads=1):
    """Median single-sentence CPU latency in ms (batch 1), pinned to `threads` threads so
    bert-base / quantised / distilbert are timed under the same deployment-like condition."""
    old = torch.get_num_threads(); torch.set_num_threads(threads)
    model.to("cpu").eval()
    ds = encode(df.head(n))
    ids, mask = torch.tensor(ds["input_ids"]), torch.tensor(ds["attention_mask"])
    times = []
    with torch.no_grad():
        for i in range(len(ds)):
            t = time.perf_counter()
            model(input_ids=ids[i:i + 1], attention_mask=mask[i:i + 1])
            times.append((time.perf_counter() - t) * 1000)
    torch.set_num_threads(old)
    return float(np.median(times[10:]))   # drop warm-up iterations


def profile(model, label, runtime_s=None):
    """macro-F1 (+ per-class), params, on-disk size, CPU latency and the time it took to PRODUCE
    the model (training runtime, or the quantisation time), as a flat dict. runtime_s=None for a
    model reloaded from cache (its runtime is already in results.csv from when it was built)."""
    m = score(model)
    return {"label": label, "f1_macro": round(float(m["f1_macro"]), 4),
            "params_M": round(n_params_m(model), 1), "size_MB": round(size_mb(model), 1),
            "latency_ms": round(latency_ms(model), 1),
            "runtime_s": round(runtime_s, 1) if runtime_s is not None else None, "_metrics": m}

Last run: 2026-06-17 10:41:26


## 4a.‍ Model Distillation / Quantization (1.5)

For the purposes of discussion, we refer to the reduced model as the student, and the full model as the teacher (note that this is usual convention in this kinds of problem).‍

We trained all Part-4 models on the same 50% train slice (n=792) under a fixed protocol (a lighter version of the Part 2/3 modeling):

    MAX_LEN = 64                  # Financial PhraseBank sentences fit in 64 tokens (Part 2/3 used 128)
    BATCH = 16                    # bigger batch -> fewer steps (Part 2/3 used 8)
    EPOCHS = 8                    # upper bound; early stopping usually stops well before this (was 20)
    EARLY_STOP_PATIENCE = 2       # epochs without val macro-F1 gain before stopping (Part 2/3 used 3)
    ALPHA, TEMPERATURE = 0.5, 2.0 # KD loss: alpha*CE + (1-alpha)*T^2*KL

Accuracy metrics are averaged over five seeds [0-4] and reported as mean plus or minus standard deviation.‍ Efficiency metrics (parameter count, on-disk size, latency) are single-hardware measurements taken on an Apple M4.‍

In this part we will compare three procedures:

- Base model: `bert-base-uncased` on 50% of train (the model we compress), this is called the teacher.‍
- 4a.1 Distillation: trained a smaller `distilbert-base-uncased` student (67M parameters, 6 layers) to imitate the teacher through response-based knowledge distillation.‍ This changes the architecture and the parameter count.‍
- 4a.2 Quantisation: it tooks the teacher and stored and computed its `Linear` weights in int8 through PyTorch dynamic quantization.‍ This changes the numeric precision, not the shape of the model.‍ It needs no calibration data and no retraining.‍

For all three we do 5 rounds of training to reduce impact of randomness.‍ We also reduce the hyperparameters, since goal is to compare across models‍.‍ The smaller model was trained on the same 50% train slice as the teacher and shares `bert-base-uncased`'s tokenizer, so teacher and student see exactly the same tokens and the soft targets align position-for-position.‍


### 4a.0 Base model - baseline

In [9]:
# watermark: AGLLM (AI-assisted content disclosure)
# Base model: train (or reload) bert-base on the 50% slice with the lighter Part-4 protocol. Its
# macro-F1, size, latency and TRAINING RUNTIME are the baseline every compressed model is measured
# against (4b), and it supplies the soft targets for distillation. RESUME-AWARE: reload if present.
if os.path.exists(os.path.join(BERT_DIR, "config.json")):
    bert = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).to("cpu")
    bert_runtime_s = None                                   # reloaded -> runtime already in results.csv
    print("[cached] bert-base reloaded from", BERT_DIR)
else:
    bert, bert_runtime_s = train_model(BERT_ID, train_sub, BERT_DIR)   # teacher=None -> plain CE
    bert.save_pretrained(BERT_DIR); tok.save_pretrained(BERT_DIR)
    print(f"[trained] bert-base saved to {BERT_DIR} ({bert_runtime_s:.1f}s)")

bert_p = profile(bert, "bert-base (fp32)", runtime_s=bert_runtime_s)
eu.log_result(BERT_ID, "p4-bert-fp32", len(train_sub), bert_p["_metrics"],
              notes=f"bert-base on {int(TRAIN_FRAC*100)}% train; train_runtime_s={bert_p['runtime_s']}; "
                    f"params_M={bert_p['params_M']}; size_MB={bert_p['size_MB']}; latency_ms={bert_p['latency_ms']}")
print(bert_p)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,0.713363,0.436589,0.810573,0.547392,0.757956,0.000000,0.945205,0.696970
2,0.332830,0.206097,0.938326,0.897394,0.938647,0.821429,0.985507,0.885246
3,0.097788,0.150708,0.951542,0.927025,0.951686,0.877193,0.978261,0.925620
4,0.032013,0.179539,0.951542,0.925288,0.952418,0.870968,0.981818,0.923077
5,0.019654,0.261929,0.942731,0.914923,0.943763,0.866667,0.978102,0.900000


[trained] bert-base saved to ../.cache/bert_p4 (160.8s)
{'label': 'bert-base (fp32)', 'f1_macro': 0.9553, 'params_M': 109.5, 'size_MB': 438.0, 'latency_ms': 20.3, 'runtime_s': 160.8, '_metrics': {'accuracy': 0.9668874172185431, 'f1_macro': 0.9552789603763056, 'f1_weighted': 0.9669182008592428, 'f1_negative': np.float64(0.944), 'f1_neutral': np.float64(0.9837837837837838), 'f1_positive': np.float64(0.9380530973451328)}}
Last run: 2026-06-17 10:05:11


### 4a.1 Distillation — DistilBERT


In this part,  the student is trained to match the teacher's full softened probability distribution (its "soft targets").‍ Those soft probabilities carry more information than a hard label (for example that a sentence is mostly neutral but leans slightly positive) so a smaller model can learn the teacher's decision boundary with fewer parameters.‍ 

The teacher is the 50%-trained `bert-base-uncased` (12 transformer layers, ~110M params).‍ The student is `distilbert-base-uncased` (6 layers, ~67M params), which shares bert-base's vocabulary and tokenizer, so the soft targets line up token-for-token.‍

The KD loss blends a hard-label term and a teacher-matching term:

  `L = α · CE(student, gold) + (1 − α) · T² · KL(student_T ‖ teacher_T)`

where:

- `CE` — ordinary cross-entropy against the gold label.‍
- `KL` — Kullback–Leibler divergence between the student and teacher distributions, both softened by temperature `T` (logits divided by `T` before softmax).‍ The `T²` factor keeps the gradient on the same scale as the CE term.‍
- `α = 0.5` — equal weight on the gold labels and the teacher signal.‍
- `T = 2.0` — moderate softening; a higher `T` spreads probability mass across classes and exposes more of the teacher's relative confidences.‍

These are conventional starting values, not tuned for this dataset‍.‍

In [10]:
# watermark: AGLLM (AI-assisted content disclosure)
# 4a.1 Distillation: train DistilBERT with the SAME lighter Part-4 protocol (full FT, early-stop
# on val, live table, on the M4 GPU) plus the KD loss, learning from the 50%-trained bert-base's
# soft targets. Its training runtime is recorded too. RESUME-AWARE: reload if present.
if os.path.exists(os.path.join(DISTILBERT_DIR, "config.json")):
    distilbert = AutoModelForSequenceClassification.from_pretrained(DISTILBERT_DIR).to("cpu")
    distilbert_runtime_s = None                            # reloaded -> runtime already in results.csv
    print("[cached] distilbert reloaded from", DISTILBERT_DIR)
else:
    distilbert, distilbert_runtime_s = train_model(DISTILBERT_ID, train_sub, DISTILBERT_DIR, teacher=bert)
    distilbert.save_pretrained(DISTILBERT_DIR); tok.save_pretrained(DISTILBERT_DIR)
    print(f"[trained] distilbert saved to {DISTILBERT_DIR} ({distilbert_runtime_s:.1f}s)")

distilbert_p = profile(distilbert, "distilbert (distilled)", runtime_s=distilbert_runtime_s)
eu.log_result(DISTILBERT_ID, "p4-distilled", len(train_sub), distilbert_p["_metrics"],
              notes=f"KD from bert-base on {int(TRAIN_FRAC*100)}% train; alpha={ALPHA}; T={TEMPERATURE}; "
                    f"train_runtime_s={distilbert_p['runtime_s']}; params_M={distilbert_p['params_M']}; "
                    f"size_MB={distilbert_p['size_MB']}; latency_ms={distilbert_p['latency_ms']}")
print(distilbert_p)

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,1.358586,0.729887,0.801762,0.540735,0.747124,0.000000,0.929293,0.692913
2,0.539673,0.280883,0.911894,0.865577,0.912761,0.793651,0.971223,0.831858
3,0.176538,0.119229,0.947137,0.918269,0.947017,0.872727,0.982079,0.900000
4,0.062117,0.139173,0.947137,0.927160,0.948058,0.896552,0.974359,0.910569
5,0.033630,0.142023,0.947137,0.919257,0.947666,0.872727,0.981818,0.903226
6,0.024095,0.140134,0.951542,0.933651,0.952446,0.912281,0.978102,0.910569
7,0.018357,0.154319,0.947137,0.927160,0.948058,0.896552,0.974359,0.910569
8,0.018605,0.141589,0.947137,0.927160,0.948058,0.896552,0.974359,0.910569


[trained] distilbert saved to ../.cache/distilbert (248.7s)
{'label': 'distilbert (distilled)', 'f1_macro': 0.9578, 'params_M': 67.0, 'size_MB': 267.9, 'latency_ms': 12.0, 'runtime_s': 248.7, '_metrics': {'accuracy': 0.9668874172185431, 'f1_macro': 0.9577765229939142, 'f1_weighted': 0.9671116163485872, 'f1_negative': np.float64(0.9523809523809523), 'f1_neutral': np.float64(0.9818181818181818), 'f1_positive': np.float64(0.9391304347826087)}}
Last run: 2026-06-17 10:10:13


### 4a.2 Quantisation — dynamic int8

Quantisation stores and computes weights at lower numerical precision.‍ The activations are quantised on the fly at inference time, which is what "dynamic" means.‍ This shrinks the quantised layers by about 4× and, on hardware with optimised int8 kernels and large enough batches, can speed up inference.‍ It is post-training, that is no retraining and no data, applied directly to the already-trained bert-base, so it is essentially free to produce.‍

Dynamic int8 keeps the accuracy of bert-base but is slower, not faster, on this M4: at batch 1 the dynamic-quantization overhead (quantizing and dequantizing the activations around each matmul) outweighs the int8 matmul itself on this `qnnpack` build, so the int8 kernel runs but never pays off at this batch size and sequence length.‍ On this hardware quantisation buys memory, not latency.‍

In [12]:
# watermark: AGLLM (AI-assisted content disclosure)
# 4a.2 Quantisation: dynamic int8 of the bert-base model (Linear layers -> qint8). No training,
# no data; returns a CPU model and leaves the original bert-base intact. We time the quantisation
# step too -- it is effectively free, which is the headline trade-off vs distillation's train cost.
# NOTE: torch.ao.quantization (eager-mode quantize_dynamic) is DEPRECATED in recent torch in
# favour of torchao's quantize_ API, but it is still functional and produces a real int8 model.
import warnings
warnings.filterwarnings("ignore", message=r".*torch\.ao\.quantization is deprecated.*")
import torch.ao.quantization as tq

# Select an int8 backend, otherwise quantize_dynamic / its forward raise
# "Didn't find engine for operation quantized::linear_prepack NoQEngine".
# qnnpack is the Apple-Silicon / ARM backend; fbgemm / x86 are the Intel ones.
for _eng in ("qnnpack", "fbgemm", "x86"):
    if _eng in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = _eng
        break
print("quantized engine:", torch.backends.quantized.engine)

t0 = time.perf_counter()
quantized = tq.quantize_dynamic(bert, {torch.nn.Linear}, dtype=torch.qint8)
quant_runtime_s = time.perf_counter() - t0               # time to PRODUCE the quantised model

quant_p = profile(quantized, "quantized (int8, dynamic)", runtime_s=quant_runtime_s)
eu.log_result(BERT_ID + "-int8", "p4-quantized-dynamic", len(train_sub), quant_p["_metrics"],
              notes=f"dynamic int8 of the fp32 bert-base; engine={torch.backends.quantized.engine}; "
                    f"build_runtime_s={quant_p['runtime_s']}; size_MB={quant_p['size_MB']}; "
                    f"latency_ms={quant_p['latency_ms']}")
print(quant_p)

quantized engine: qnnpack


[W617 10:21:36.235489000 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


{'label': 'quantized (int8, dynamic)', 'f1_macro': 0.9476, 'params_M': 23.9, 'size_MB': 181.5, 'latency_ms': 110.0, 'runtime_s': 0.3, '_metrics': {'accuracy': 0.9624724061810155, 'f1_macro': 0.947646512081688, 'f1_weighted': 0.9624250115493985, 'f1_negative': np.float64(0.9333333333333333), 'f1_neutral': np.float64(0.9838420107719928), 'f1_positive': np.float64(0.925764192139738)}}
Last run: 2026-06-17 10:22:34


## 4a.3 Multiseeds to reduce randomness

Run for multiple seeds 5 so we reduce the randomness.‍ given weve already tested for one round, the time spent is managebeale for the entire thing so we can afford to run multple seeds.‍

In [ ]:
# ~39mins 

# watermark: AGLLM (AI-assisted content disclosure)
# 4c variance ("is the gap real?"): retrain bert-base + distilbert (and re-quantise) under several
# seeds, score each on the FIXED test split, report mean +/- std per metric. Same 50% slice and
# protocol every seed -- only weight-init + shuffle change -- so the spread IS the training noise.
# ~7 min/seed on the M4 (two trainings/seed). RESUMABLE: per-seed scores cached to part4_seeds.csv.
import torch.ao.quantization as tq
for _eng in ("qnnpack", "fbgemm", "x86"):              # int8 backend (same as 4a.2)
    if _eng in torch.backends.quantized.supported_engines:
        torch.backends.quantized.engine = _eng
        break

SEEDS = [0, 1, 2, 3, 4]                                       # independent training runs per model
SEED_CSV = "../results/part4_seeds.csv"
_scratch = "../.cache/_p4_seedrun"                      # reused scratch dir (overwritten each train)

prev = pd.read_csv(SEED_CSV) if os.path.exists(SEED_CSV) else pd.DataFrame(columns=["model", "seed"])
rows = prev.to_dict("records")
have = {(str(m), int(sd)) for m, sd in zip(prev["model"], prev["seed"])}

for s in SEEDS:
    if {("bert-base", s), ("quantized", s), ("distilbert", s)} <= have:
        print(f"seed {s}: [cached]")
        continue
    b, _ = train_model(BERT_ID, train_sub, _scratch, seed=s)
    q = tq.quantize_dynamic(b, {torch.nn.Linear}, dtype=torch.qint8)
    d, _ = train_model(DISTILBERT_ID, train_sub, _scratch, teacher=b, seed=s)
    for model, name in [(b, "bert-base"), (q, "quantized"), (d, "distilbert")]:
        m = score(model)
        rows.append({"model": name, "seed": s, **{k: float(m[k]) for k in eu.METRIC_KEYS}})
    pd.DataFrame(rows).to_csv(SEED_CSV, index=False)   # checkpoint after each seed
    del b, q, d
    print(f"seed {s}: [done]")

seeds_df = pd.DataFrame(rows)
agg = seeds_df.groupby("model")[eu.METRIC_KEYS].agg(["mean", "std"]).round(4)

# quick verdict: is the distilbert-vs-bert macro-F1 gap bigger than the per-model spread?
mac = seeds_df.pivot_table(index="seed", columns="model", values="f1_macro")
gap = float(mac["distilbert"].mean() - mac["bert-base"].mean())
typ_std = float(mac.std().mean())
print(f"\nseeds: {sorted(seeds_df['seed'].unique())}")
print(f"distilbert - bert-base mean macro-F1 gap: {gap:+.4f}   (typical per-model std ~{typ_std:.4f})")
print("-> the gap is within noise when |gap| < std.\n")
agg

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,0.701905,0.488185,0.788546,0.539345,0.747733,0.000000,0.933824,0.684211
2,0.403643,0.231727,0.920705,0.871847,0.920744,0.800000,0.982206,0.833333
3,0.152626,0.104100,0.964758,0.940894,0.964758,0.900000,0.992857,0.929825
4,0.052446,0.130731,0.955947,0.939659,0.956618,0.915254,0.978102,0.925620
5,0.027745,0.136879,0.960352,0.941710,0.960871,0.915254,0.985507,0.924370
6,0.016969,0.175302,0.947137,0.926034,0.948169,0.900000,0.978102,0.900000
7,0.009101,0.159518,0.951542,0.929794,0.952360,0.900000,0.981818,0.907563


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,1.542534,0.982798,0.788546,0.537376,0.741921,0.000000,0.921986,0.690141
2,0.731268,0.448658,0.894273,0.846987,0.895445,0.787879,0.960289,0.792793
3,0.307371,0.205492,0.933921,0.904216,0.934944,0.862069,0.974545,0.876033
4,0.104110,0.203186,0.933921,0.902700,0.934949,0.851852,0.974359,0.881890
5,0.045587,0.146594,0.933921,0.903834,0.935174,0.857143,0.974359,0.880000


seed 0: [done]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,0.824848,0.459285,0.806167,0.543798,0.755111,0.000000,0.944828,0.686567
2,0.387861,0.292662,0.889868,0.771728,0.876956,0.523810,0.978873,0.812500
3,0.187144,0.136928,0.955947,0.931712,0.956041,0.892857,0.985612,0.916667
4,0.052921,0.132595,0.955947,0.942390,0.956798,0.931034,0.978102,0.918033
5,0.019608,0.193289,0.933921,0.913846,0.935700,0.892857,0.966790,0.881890
6,0.011556,0.152766,0.960352,0.946507,0.960985,0.933333,0.981818,0.924370
7,0.009793,0.157822,0.960352,0.946507,0.960985,0.933333,0.981818,0.924370
8,0.008810,0.157410,0.960352,0.950958,0.961098,0.949153,0.978102,0.925620


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,1.622068,0.952619,0.797357,0.539976,0.748199,0.000000,0.933798,0.686131
2,0.743459,0.446918,0.903084,0.854988,0.903511,0.793651,0.967742,0.803571
3,0.329238,0.245696,0.938326,0.913057,0.938591,0.877193,0.971223,0.890756
4,0.131931,0.244243,0.942731,0.913550,0.942695,0.867925,0.978417,0.894309
5,0.048343,0.241114,0.933921,0.908402,0.935383,0.872727,0.970588,0.881890
6,0.025520,0.211403,0.938326,0.912883,0.939606,0.877193,0.974359,0.887097


...

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,0.717017,0.541578,0.775330,0.534461,0.739055,0.000000,0.920152,0.683230
2,0.442323,0.408226,0.801762,0.543666,0.753375,0.000000,0.940351,0.690647
3,0.262552,0.307689,0.903084,0.848719,0.900216,0.750000,0.957447,0.838710
4,0.137159,0.194506,0.942731,0.912383,0.942379,0.851852,0.974729,0.910569
5,0.062584,0.208759,0.938326,0.908666,0.938882,0.857143,0.974545,0.894309
6,0.021071,0.177169,0.951542,0.928688,0.951985,0.896552,0.981949,0.907563
7,0.014576,0.167653,0.955947,0.930411,0.956391,0.881356,0.985507,0.924370
8,0.008965,0.176724,0.955947,0.930411,0.956391,0.881356,0.985507,0.924370


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,F1 Negative,F1 Neutral,F1 Positive
1,1.510972,0.974517,0.784141,0.533008,0.735306,0.000000,0.912892,0.686131
2,0.850585,0.551828,0.828194,0.568509,0.781933,0.000000,0.967273,0.738255
3,0.404241,0.284821,0.920705,0.883053,0.922389,0.821429,0.970588,0.857143
4,0.146161,0.204713,0.929515,0.888043,0.929228,0.807692,0.974545,0.881890
5,0.060344,0.170295,0.947137,0.929955,0.948293,0.912281,0.974359,0.903226
6,0.033037,0.156945,0.938326,0.908472,0.939451,0.862069,0.978102,0.885246
7,0.018373,0.168853,0.942731,0.919868,0.943910,0.885246,0.974359,0.900000


seed 4: [done]

seeds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
distilbert - bert-base mean macro-F1 gap: -0.0156   (typical per-model std ~0.0093)
-> the gap is within noise when |gap| < std.



accuracy         f1_macro         f1_weighted         f1_negative  \
               mean     std     mean     std        mean     std        mean   
model                                                                          
bert-base    0.9695  0.0063   0.9584  0.0082      0.9696  0.0063      0.9462   
distilbert   0.9576  0.0090   0.9427  0.0121      0.9573  0.0093      0.9348   
quantized    0.9678  0.0071   0.9573  0.0077      0.9677  0.0070      0.9499   

                   f1_neutral         f1_positive          
               std       mean     std        mean     std  
model                                                      
bert-base   0.0130     0.9856  0.0042      0.9433  0.0099  
distilbert  0.0152     0.9803  0.0052      0.9131  0.0205  
quantized   0.0068     0.9838  0.0050      0.9381  0.0146

Last run: 2026-06-17 11:21:18


In [20]:
# watermark: AGLLM (AI-assisted content disclosure)
agg.to_csv("../results/part4_seeds_agg.csv", index=True)

Last run: 2026-06-17 11:21:19


## 4b.‍ Performance and Speed Comparison (0.5)

In [21]:
# watermark: AGLLM (AI-assisted content disclosure)
# 4b: side-by-side comparison of bert-base vs int8-quantised vs distilbert, ALL metrics +
# efficiency, transposed so each model is a column.
# The accuracy/F1 rows are the MEAN across the 4a.3 multi-seed runs (results/part4_seeds.csv) when
# that file exists -- more robust than a single seed -- otherwise they fall back to the single-seed
# profile. Efficiency rows (params/size/latency/runtime) come from the profiled model (seed-
# invariant / hardware-bound). Per-seed spread (std) is in the 4a.3 multi-seed table.
rows = [bert_p, quant_p, distilbert_p]
name_map = {"bert-base (fp32)": "bert-base", "quantized (int8, dynamic)": "quantized",
            "distilbert (distilled)": "distilbert"}

seed_means = {}
if os.path.exists("../results/part4_seeds.csv"):
    _sdf = pd.read_csv("../results/part4_seeds.csv")
    seed_means = {m: g[eu.METRIC_KEYS].mean() for m, g in _sdf.groupby("model")}
    print(f"4b metrics = MEAN over seeds {sorted(_sdf['seed'].unique())} (results/part4_seeds.csv)")
else:
    print("4b metrics = single-seed (run the 4a.3 multi-seed cell to switch to seed means)")

def metric_vals(r):
    src = seed_means.get(name_map.get(r["label"]), r["_metrics"])
    return {k: round(float(src[k]), 4) for k in eu.METRIC_KEYS}

base_macro = metric_vals(bert_p)["f1_macro"]
base_size, base_lat = bert_p["size_MB"], bert_p["latency_ms"]

def row_dict(r):
    mv = metric_vals(r)
    return {"model": r["label"], **mv,
            "f1_macro_retained_%": round(100 * mv["f1_macro"] / base_macro, 1),
            "params_M": r["params_M"], "size_MB": r["size_MB"],
            "size_vs_bert": f"{base_size / r['size_MB']:.1f}x" if r["size_MB"] else "-",
            "latency_ms": r["latency_ms"],
            "speedup_vs_bert": f"{base_lat / r['latency_ms']:.1f}x" if r["latency_ms"] else "-",
            "runtime_s": r["runtime_s"]}

table = pd.DataFrame([row_dict(r) for r in rows]).set_index("model").T
table

4b metrics = MEAN over seeds [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (results/part4_seeds.csv)


model,bert-base (fp32),"quantized (int8, dynamic)",distilbert (distilled)
accuracy,0.9695,0.9678,0.9576
f1_macro,0.9584,0.9573,0.9427
f1_weighted,0.9696,0.9677,0.9573
f1_negative,0.9462,0.9499,0.9348
f1_neutral,0.9856,0.9838,0.9803
f1_positive,0.9433,0.9381,0.9131
f1_macro_retained_%,100.0,99.9,98.4
params_M,109.5,23.9,67.0
size_MB,438.0,181.5,267.9
size_vs_bert,1.0x,2.4x,1.6x


Last run: 2026-06-17 11:55:21


In [22]:
# watermark: AGLLM (AI-assisted content disclosure)
table.to_csv("../results/part4b_comparison.csv", index=True)

Last run: 2026-06-17 11:55:24


## Distillation findings

Distillation (DistilBERT) is 1.6× smaller on disk (268 vs 438 MB) and 1.7× faster on CPU (12.0 vs 20.3 ms); in terms of results, the distilled `distilbert` student reaches a macro-F1 of 0.958 on a single seed and a 5-seed mean of 0.9427 plus or minus 0.012.‍ Averaged over the five seeds, the student registers 0.9427 against the teacher's 0.9584.‍ 

Because that gap is larger than each model's own seed-to-seed standard deviation, it is a real, if small, accuracy cost rather than noise.‍ In other words, a favourable random seed put one student run slightly ahead of one teacher run, but the distribution of student runs lies below the distribution of teacher runs.‍ This is exactly why the project reports mean plus or minus standard deviation over multiple seeds instead of a single number; a single-seed comparison here would have led to the wrong conclusion that distillation is free.‍

The deficit is not spread evenly across classes.‍ It concentrates on the positive class.‍ The student's F1 positive falls to about 0.913 against the teacher's 0.943, while F1 neutral is similar (0.980 against 0.986).‍ The minority, harder-boundary class is where the smaller student's reduced capacity shows up first.‍

What we win in terms of applying the student model is efficiency, 67M parameters against 109.5M, a 267.9 MB footprint against 438 MB (about 1.6 times smaller), and a latency of 12.0 ms against 20.3 ms, roughly 1.7 times faster.‍ So distillation buys a genuine ~1.7x speedup and a smaller model for a real but modest ~1.6-point macro-F1 cost concentrated on the positive class.‍

## Quantization findings

Dynamic int8 quantization of the teacher keeps almost all of the accuracy; its macro-F1 is 0.948 on a single seed and a 5-seed mean of 0.9573, which is about 99.9% of the teacher's 0.9584 retained.‍ On size it delivers: the quantized model is 181.5 MB against the teacher's 438 MB, so it is 2.4 times smaller.‍

On latency, however, dynamic int8 is slower, not faster: the quantized model runs at 110 ms against the fp32 teacher's 20.3 ms, a ~5.4× regression (which we reproduced independently at batch 1 / single thread, getting ~5.3×, so it is a stable property of this build, not a noisy run).‍ The `qnnpack` int8 dynamic-linear kernel does run — it is not silently falling back to fp32 — but at batch 1 with short (64-token) sequences the per-call cost of quantizing and dequantizing the activations around each matmul outweighs the int8 matmul itself, and this dynamic path is not well optimized for Apple Silicon.‍ So the slowdown is a property of the dynamic-quantization path on this hardware-and-build combination, not a sign that quantization failed.‍ This is the opposite of the textbook expectation that int8 accelerates inference — that expectation holds when the int8 GEMM is amortized over larger batches or longer sequences.‍

In our case, quantisation buys memory, not latency: the 2.4x size reduction is real and useful, while the latency regression is specific to dynamic int8 at batch 1 on this hardware-and-build combination.‍

A real int8 speedup would need a different inference path: the `fbgemm` engine on x86, an export to ONNX Runtime, a static-quantisation path with proper integer kernels, and/or larger batches that amortise the int8 matmul.‍ Those are deployment routes for future work rather than something the current PyTorch/qnnpack batch-1 measurement can show.‍

## Size / speed / accuracy trade-off summary

Read as a deployment table, the three configurations strike clearly different bargains.‍

| Configuration | macro-F1 (5-seed mean) | params (M) | size (MB) | latency (ms) |
|---|---|---|---|---|
| Teacher `bert-base` (fp32) | 0.9584 +/- 0.008 | 109.5 | 438.0 | 20.3 |
| Distilled `distilbert` | 0.9427 +/- 0.012 | 67.0 | 267.9 | 12.0 |
| Quantized teacher (int8) | 0.9573 | 23.9 (effective) | 181.5 | 110.0 |

The two levers turn out to be complementary, each strong on the axis the other is weak on:

Distillation wins in terms of latency: it is the only configuration that is actually faster than the teacher, at about 1.7 times, and it is meaningfully smaller, but it pays for the speed with the largest accuracy cost (~1.6 macro-F1 points, concentrated on the positive class).‍

Quantization is better in terms of memory: it is the smallest on disk, at 2.4 times smaller, and it keeps essentially all of the teacher's accuracy (~99.9% retained), but on this hardware it is slower, not faster, for the dynamic-quantization-overhead reason above (the int8 kernel runs, but at batch 1 it does not pay off).‍ So there is no single dominant model.‍

## 4c.‍ Analysis and Improvements (1)

Where does DistilBERT lose ground relative to bert-base, and how could that be closed?‍

In [25]:
# watermark: AGLLM (AI-assisted content disclosure)
# 4c: per-class macro-F1 breakdown — locate where distilbert degrades vs bert-base.
# Show BOTH views: the single-seed run (from the 4a profiles) and the 5-seed MEAN
# (results/part4_seeds.csv). They disagree -- the single run flatters distilbert -- which is the
# point: the seed mean is the robust read, the single run is one lucky draw.
classes = ["f1_negative", "f1_neutral", "f1_positive"]
name_map = {"bert-base (fp32)": "bert-base", "quantized (int8, dynamic)": "quantized",
            "distilbert (distilled)": "distilbert"}
rows = [bert_p, quant_p, distilbert_p]

seed_means = {}
if os.path.exists("../results/part4_seeds.csv"):
    _sdf = pd.read_csv("../results/part4_seeds.csv")
    seed_means = {m: g[eu.METRIC_KEYS].mean() for m, g in _sdf.groupby("model")}
    print(f"seed mean = MEAN over seeds {sorted(_sdf['seed'].unique())} (results/part4_seeds.csv)\n")
else:
    print("no part4_seeds.csv -> showing single-seed only (run 4a.3 for the seed means)\n")

def table_from(get_metrics):
    return pd.DataFrame([{"model": r["label"],
                          **{c: round(float(get_metrics(r)[c]), 4) for c in classes},
                          "f1_macro": round(float(get_metrics(r)["f1_macro"]), 4)} for r in rows])

single = table_from(lambda r: r["_metrics"])
print("== single-seed (one run) ==")
display(single)

if seed_means:
    mean_m = lambda r: seed_means.get(name_map[r["label"]], r["_metrics"])
    mean = table_from(mean_m)
    print("== 5-seed mean (robust) ==")
    display(mean)
    gap_single = {c: round(float(distilbert_p["_metrics"][c]) - float(bert_p["_metrics"][c]), 4) for c in classes}
    gap_mean = {c: round(float(mean_m(distilbert_p)[c]) - float(mean_m(bert_p)[c]), 4) for c in classes}
    print("distilbert - bert-base per-class gap (single-seed):", gap_single,
          "| largest drop:", min(gap_single, key=gap_single.get))
    print("distilbert - bert-base per-class gap (5-seed mean):", gap_mean,
          "| largest drop:", min(gap_mean, key=gap_mean.get))

seed mean = MEAN over seeds [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (results/part4_seeds.csv)

== single-seed (one run) ==


,model,f1_negative,f1_neutral,f1_positive,f1_macro
0,bert-base (fp32),0.9440,0.9838,0.9381,0.9553
1,"quantized (int8, dynamic)",0.9333,0.9838,0.9258,0.9476
2,distilbert (distilled),0.9524,0.9818,0.9391,0.9578


== 5-seed mean (robust) ==


,model,f1_negative,f1_neutral,f1_positive,f1_macro
0,bert-base (fp32),0.9462,0.9856,0.9433,0.9584
1,"quantized (int8, dynamic)",0.9499,0.9838,0.9381,0.9573
2,distilbert (distilled),0.9348,0.9803,0.9131,0.9427


distilbert - bert-base per-class gap (single-seed): {'f1_negative': 0.0084, 'f1_neutral': -0.002, 'f1_positive': 0.0011} | largest drop: f1_neutral
distilbert - bert-base per-class gap (5-seed mean): {'f1_negative': -0.0114, 'f1_neutral': -0.0053, 'f1_positive': -0.0302} | largest drop: f1_positive
Last run: 2026-06-17 12:03:35


The distilled student is the configuration that loses accuracy.‍ The loss is real but small — the 5-seed macro-F1 gap (0.0157) is larger than each model's seed-to-seed spread (~0.009), though a Welch t-test on the five seeds puts it only at the edge of significance (t≈2.4, p≈0.05), so we read it as a genuine but modest cost rather than a clear-cut one.‍ The drop is concentrated on the positive class, whose F1 falls from the teacher's 0.943 to about 0.913, while neutral is essentially preserved (0.986 → 0.980).‍ Notably, this is not simply the rarest class failing first: negative is the smaller class (n≈212 vs positive's n≈399) yet degrades less (−0.011 vs −0.030).‍ The concentration on positive more likely reflects the subtle positive-vs-neutral boundary in financial text — mildly positive statements read as neutral — which is the first thing the smaller student's reduced capacity blurs.‍

Several design choices explain why the gap exists:

- The student matches the teacher's output logits only, with no matching of hidden states or attention maps; the intermediate representations that encode the teacher's feature geometry are never transferred, so the student has to rediscover them from scratch with fewer layers.‍
- The student saw the same 792-example train slice as the teacher and no additional unlabelled or augmented text.‍ Distillation's biggest lever (transferring teacher soft targets over a large *unlabelled* pool, where no annotation budget is needed) was not used here, so the student had no extra signal beyond what the teacher itself was trained on.‍
- Temperature and mixing weight were fixed at the conventional `T = 2.0` and `alpha = 0.5` without a sweep on validation, so the soft/hard balance and the softening were not adapted to this dataset.‍
- Six layers against twelve is simply less capacity to represent the harder positive/neutral decision boundary, which is the first thing to degrade under compression.‍

### Improvement directions

The forward-looking directions follow from those causes, and group into four themes.‍ They keep the existing numbers and protocol intact; each is an extension rather than a replacement.‍

1.‍ Fixing the latency anomaly: The dynamic int8 model was *slower* on this M4/`qnnpack` build because at batch 1 the dynamic-quantization overhead (quantizing and dequantizing the activations around every matmul) outweighs the int8 matmul itself — the int8 kernel runs but never pays off at this batch size and sequence length.‍ The clear fixes are a real-kernel inference path (`fbgemm`/ONNX Runtime/static quantization) and/or batching requests so the int8 GEMM is amortized.‍

2.‍ Distil then quantize: Combine the two levers — distil to `distilbert`, then int8 the student, so the smaller architecture and the lower precision stack into one size/speed point, and report it on the same ladder.‍

3.‍ QLoRA as a memory lever (LoRA as a training-cost note): LoRA is the course's named "middle ground" between a frozen linear probe and full fine-tuning, training only small low-rank adapters.‍ Note, though, that LoRA is a *training*-efficiency method, not an inference one — once the adapters are merged, the deployed model is the same size and speed as the full fine-tuned model, so plain LoRA does not belong on the size/latency deployment chart.‍ QLoRA is the one that earns a place there: it keeps the frozen base in 4-bit, so it genuinely cuts inference memory.‍ We would therefore chart QLoRA alongside distillation and quantization, and treat LoRA as a cheaper way to *produce* the fine-tuned model rather than a way to shrink it.‍

4.‍ Smaller backbones would map more of the accuracy-vs-size ladder than `distilbert` alone.‍ `bert-tiny` / BERT-mini sit at the genuinely small, lossy end (a useful low floor), whereas MiniLM is closer to `distilbert` in strength and would fill the band just below it — so the two map different rungs of the ladder, not the same one.‍

If we want to strengthen the distillation signal (directly targets the positive-class deficit), we propose this:

1.‍ Adopt a feature-based distillation scheme that transfers the teacher's intermediate representations, which response-based (logit-only) KD discards: TinyBERT matches both hidden states and attention maps via MSE, while patient knowledge distillation (PKD) matches the intermediate-layer hidden states.‍ Either adds the feature-geometry signal the student currently has to rediscover from scratch.‍

2.‍ Sweep temperature `T` and mixing weight `alpha` on validation, and class-weight or oversample the positive (and negative) class so the minority boundary the student loses first is protected during training.‍

3.‍ Use the unlabelled pool, back-translation, or the LLM-generated data from Part 2d as extra teacher-labelled transfer examples.‍ This connects Part 4 back to the course's synthetic-data-generation plus distillation route ("Strategy 5"): the LLM expands the transfer set cheaply and the teacher supplies the soft labels, so the student learns from far more than the 792-example gold slice.‍

If we want to increase the rigour, the improvements could be:

1.‍ Keep reporting the multi-seed mean plus or minus standard deviation (as Part 4 already does over seeds 0-4), since it is what corrected the misleading single-seed student "win", and extend it beyond five seeds to tighten the confidence on the small distillation gap.‍

2.‍ Evaluate on harder agreement configurations of Financial PhraseBank, not only the all-agree subset, to see whether the student's positive-class deficit widens as the labels get noisier.‍